# Train and export an expert package

This NB-003 workflow trains one full-MLP expert at every converted layer while the Qwen3 anchor remains immutable. It exports only expert tensors and package metadata; router fitting remains NB-004. For the TRN-003 two-expert joint alternative, use `LocalMiniMoETrainer` as demonstrated by `examples/verify_local_mini_moe.py`; package export remains an explicit later step.

## Resolve configuration

One mapping records the model, dataset, training, router-feature, and package choices used by the workflow.

In [ ]:
import os
import platform
from pathlib import Path
from tempfile import TemporaryDirectory

import accelerate
import safetensors
import torch
import transformers
from transformers import Qwen3Config, Qwen3ForCausalLM, TrainingArguments

from anchored_moe import (
    AnchoredMoEConfig,
    ExpertPackageExportBinding,
    ExpertPackageExportConfig,
    ForcedExpertBinding,
    ForcedPathExpertTrainer,
    FullMLPExpert,
    RoutedMLP,
    export_full_mlp_expert_package,
)
from anchored_moe.backends import Qwen3Backend
from anchored_moe.evaluation import fingerprint_module
from anchored_moe.utils import sha256_json

smoke_test = os.environ.get("ANCHORED_MOE_SMOKE_TEST", "1") == "1"
resolved_config = {
    "feature_id": "NB-003",
    "base_model": {
        "model_id": "local-random-qwen3",
        "revision": "synthetic-seed-13",
        "tokenizer_revision": "synthetic-seed-13",
        "converted_layers": [2, 3],
        "dtype": "float32",
    },
    "dataset": {
        "dataset_id": "synthetic-local-code",
        "revision": "1",
        "split": "train",
        "filters": [],
    },
    "training": {
        "strategy": "forced_path",
        "expert_id": "code:0",
        "max_steps": 1 if smoke_test else 20,
        "learning_rate": 1e-2 if smoke_test else 1e-4,
        "seed": 41,
    },
    "router_features": {
        "feature_type": "identity",
        "input": "pre_mlp_hidden_state",
        "output_size": 16,
        "version": 1,
    },
    "package": {
        "specialization_name": "code",
        "specialization_version": "1.0.0",
    },
}
resolved_config

## Load data and model

The checked workflow uses a deterministic local model and token dataset, so its invariants execute without network or accelerator access.

In [ ]:
torch.manual_seed(13)
model = Qwen3ForCausalLM(
    Qwen3Config(
        vocab_size=64,
        hidden_size=16,
        intermediate_size=32,
        num_hidden_layers=4,
        num_attention_heads=4,
        num_key_value_heads=2,
        head_dim=4,
        max_position_embeddings=32,
    )
).eval()
base_weight_sha256 = fingerprint_module(model)
training_dataset = []
for values in ((1, 7, 11, 3), (2, 5, 13, 9)):
    tokens = torch.tensor(values, dtype=torch.long)
    with torch.no_grad():
        anchor_logits = model(input_ids=tokens.unsqueeze(0)).logits.squeeze(0)
    training_dataset.append(
        {
            "input_ids": tokens,
            "attention_mask": torch.ones_like(tokens),
            "labels": tokens.clone(),
            "anchor_logits": anchor_logits.detach().clone(),
        }
    )

backend = Qwen3Backend()
backend.convert(
    model,
    AnchoredMoEConfig(
        converted_layers=resolved_config["base_model"]["converted_layers"]
    ),
)
training_bindings = []
anchor_fingerprints = {}
expert_initial_state = {}
for layer_index in resolved_config["base_model"]["converted_layers"]:
    wrapper = model.model.layers[layer_index].mlp
    assert isinstance(wrapper, RoutedMLP)
    anchor_fingerprints[layer_index] = fingerprint_module(wrapper.anchor_mlp)
    expert = FullMLPExpert.from_anchor(wrapper.anchor_mlp)
    expert_initial_state[layer_index] = {
        name: tensor.detach().clone()
        for name, tensor in expert.state_dict().items()
    }
    training_bindings.append(
        ForcedExpertBinding(
            layer_index,
            resolved_config["training"]["expert_id"],
            expert,
        )
    )

## Run framework operation

The constrained Trainer optimizes exactly the attached experts. Package publication is a separate explicit operation.

In [ ]:
temporary_workspace = TemporaryDirectory(prefix="anchored-moe-nb003-")
workspace = Path(temporary_workspace.name)
trainer = ForcedPathExpertTrainer(
    model=model,
    backend=backend,
    expert_bindings=training_bindings,
    args=TrainingArguments(
        output_dir=workspace / "trainer",
        max_steps=resolved_config["training"]["max_steps"],
        per_device_train_batch_size=1,
        learning_rate=resolved_config["training"]["learning_rate"],
        optim="adamw_torch",
        save_strategy="no",
        logging_strategy="no",
        disable_tqdm=True,
        use_cpu=True,
        dataloader_pin_memory=False,
        seed=resolved_config["training"]["seed"],
    ),
    train_dataset=training_dataset,
)
train_result = trainer.train()
trainer.forced_path_report

In [ ]:
package = export_full_mlp_expert_package(
    workspace / "code-expert",
    bindings=[
        ExpertPackageExportBinding(
            binding.layer_index,
            binding.expert_id,
            binding.expert,
        )
        for binding in training_bindings
    ],
    config=ExpertPackageExportConfig(
        specialization_name=resolved_config["package"]["specialization_name"],
        specialization_version=resolved_config["package"]["specialization_version"],
        base_model_repository=resolved_config["base_model"]["model_id"],
        base_model_revision=resolved_config["base_model"]["revision"],
        base_config_sha256=sha256_json(model.config.to_dict()),
        base_weight_sha256={"in-memory-anchor-state": base_weight_sha256},
        tokenizer_repository=resolved_config["base_model"]["model_id"],
        tokenizer_revision=resolved_config["base_model"]["tokenizer_revision"],
        base_model_dtype=resolved_config["base_model"]["dtype"],
        runtime_versions={
            "accelerate": accelerate.__version__,
            "kernels": "torch-default-cpu",
            "peft": "not-installed",
            "python": platform.python_version(),
            "safetensors": safetensors.__version__,
            "torch": torch.__version__,
            "transformers": transformers.__version__,
        },
        training_dataset=resolved_config["dataset"],
        router_feature_specification=resolved_config["router_features"],
        expert_parameterization_config={
            "initialization": "anchor_copy",
            "training_strategy": resolved_config["training"]["strategy"],
            "optimizer": "adamw_torch",
            "max_steps": train_result.global_step,
        },
    ),
)
package.identity

## Validate invariants and summarize

The final checks prove expert updates, byte-exact converted anchors, complete payload ownership, expert-only serialization, and strict package reload.

In [ ]:
experts_changed = all(
    any(
        not torch.equal(
            tensor,
            expert_initial_state[binding.layer_index][name],
        )
        for name, tensor in binding.expert.state_dict().items()
    )
    for binding in training_bindings
)
anchor_exact = all(
    fingerprint_module(model.model.layers[layer_index].mlp.anchor_mlp)
    == anchor_fingerprints[layer_index]
    for layer_index in resolved_config["base_model"]["converted_layers"]
)
expected_binding_keys = [
    (layer_index, resolved_config["training"]["expert_id"])
    for layer_index in resolved_config["base_model"]["converted_layers"]
]
assert experts_changed
assert anchor_exact
assert [binding.key for binding in package.payload_bindings] == expected_binding_keys
assert package.manifest.router_architecture == "none"
assert package.manifest.router_files == ()
package.validate_live_files()
validation_summary = {
    "feature_id": resolved_config["feature_id"],
    "steps": train_result.global_step,
    "converted_layers": list(package.manifest.converted_layers),
    "trainable_parameter_count": trainer.forced_path_report.trainable_parameter_count,
    "expert_payload_count": len(package.manifest.expert_weight_files),
    "anchor_exact": anchor_exact,
    "experts_changed": experts_changed,
    "router_included": False,
    "manifest_sha256": package.identity.manifest_sha256,
}
print("NB-003 validation passed", validation_summary)